In [1]:
from vcm_model import  *

In [2]:
##评估指标说明：
##每个指标对应的list按照如下顺序输入：
##第一个元素为指标类型，type1:原子指标；type2:策略不影响分母；type3:策略影响分母
##第二个元素为指标分子（原子指标即为指标本身）
##第三个元素为分子的协变量，list形式，无协变量则置空
##第四个元素为指标分母
##第五个元素为分母的协变量，list形式，无协变量则置空
eval_dict={
    '冒泡数':['type1','bubble_cnt',[]],
    '发单数':['type1','compete_call_cnt_hxz',[]],
    '完单数':['type1','hxz_finish_cnt',['compete_call_cnt_hxz','valid_tsh']],
    '应答率':['type3','hxz_answer_cnt',['compete_call_cnt_hxz','valid_tsh'],'compete_call_cnt_hxz',['compete_call_cnt_hxz','valid_tsh']],
    '成交率':['type3','hxz_finish_cnt',['compete_call_cnt_hxz','valid_tsh'],'compete_call_cnt_hxz',['compete_call_cnt_hxz','valid_tsh']],
    '应答后司机取消率(含改派)':['type3','hxz_grab_after_dri_cancel_cnt',['compete_call_cnt_hxz','valid_tsh'],'hxz_answer_cnt',['compete_call_cnt_hxz','valid_tsh']],
    '应答后司机取消率(不含改派)':['type3','hxz_grab_after_dri_cancel_cnt_without_reassign',['compete_call_cnt_hxz','valid_tsh'],'hxz_answer_cnt',['compete_call_cnt_hxz','valid_tsh']],
    '应答后乘客取消率':['type3','hxz_grab_after_pas_cancel_cnt',['compete_call_cnt_hxz','valid_tsh'],'hxz_answer_cnt',['compete_call_cnt_hxz','valid_tsh']],
    '自营完单数':['type1','hxz_zy_finish_cnt',['compete_call_cnt_hxz','valid_tsh']],
    '宏鹄完单数':['type1','hxz_hh_finish_cnt',['compete_call_cnt_hxz','valid_tsh']],
    'API完单数':['type1','hxz_api_finish_cnt',['compete_call_cnt_hxz','valid_tsh']]
}

In [3]:
##评估指标说明：
##每个指标对应的list按照如下顺序输入：
##第一个元素为指标类型，type1:原子指标；type2:策略不影响分母；type3:策略影响分母
##第二个元素为指标分子（原子指标即为指标本身）
##第三个元素为分子的协变量，list形式，无协变量则置空
##第四个元素为指标分母
##第五个元素为分母的协变量，list形式，无协变量则置空
# eval_dict={
#     'GMV':['type1','gmv',['发单量','TSH'],],
#     '单均GMV':['type2','gmv',['TSH'],'发单量'],
#     '应答后取消率':['type3','应答后取消量',[],'应答量',['发单量','TSH']],
# }

In [4]:
##数据格式说明：
##city_id：城市id，必须项
##tdate：实验日期，必须项
##exp_group：实验分组情况，枚举值treatment_group、control_group，必须项
##minf：时间片标识，必须项
##以上必须项的字段名称和实验分组名称，需与demo数据保持一致
df = pd.read_csv('data4.csv')
def chenge_group_name(x):
    return x + '_group'
df['exp_group'] = df['ab_group'].apply(chenge_group_name)
df.head()

,city_id,tdate,minf,ab_group,hxz_finish_cnt,hxz_answer_cnt,hxz_answer_cnt_without_reassign,hxz_grab_after_dri_cancel_cnt,hxz_grab_after_dri_cancel_cnt_without_reassign,hxz_grab_after_pas_cancel_cnt,hxz_grab_after_pas_cancel_cnt_without_reassign,hxz_gmv_amt,hxz_zy_answer_cnt,hxz_zy_finish_cnt,hxz_hh_answer_cnt,hxz_hh_finish_cnt,hxz_api_finish_cnt,pbd_answer_cnt,pbd_finish_cnt,pbd_zy_answer_cnt,pbd_zy_finish_cnt,pbd_hh_answer_cnt,pbd_hh_finish_cnt,pbd_api_finish_cnt,thlm_answer_cnt,thlm_finish_cnt,thlm_zy_answer_cnt,thlm_zy_finish_cnt,thlm_hh_answer_cnt,thlm_hh_finish_cnt,thlm_api_finish_cnt,juhe_zyhh_finish_cnt,juhe_zy_finish_cnt,juhe_zy_gmv_amt,compete_call_cnt_unity,compete_call_cnt_hxz,compete_call_cnt_pbd,compete_call_cnt_zy,compete_call_cnt_hh,compete_call_cnt_zyhh,compete_call_cnt_sf,compete_call_cnt_hxz_zy,compete_call_cnt_hxz_hh,compete_call_cnt_hxz_sf,compete_call_cnt_pbd_zy,compete_call_cnt_pbd_hh,compete_call_cnt_pbd_sf,bubble_cnt,valid_tsh,exp_group
0,39,2025-03-18,5,control,65,79,76,1,0,13,11,970.470000,13,7,1,1,22,41,36,2,2,0,0,0,9,8,9,8,0,0,0,18,17,294.23,423,95,124,398,100,400,200,90,87,92,106,12,106,131,22.416667,control_group
1,39,2025-03-18,6,treatment,103,143,128,19,1,21,18,1916.769388,48,27,11,3,36,81,62,15,12,3,0,0,46,33,45,32,0,0,1,74,71,1296.25,955,170,244,928,197,931,364,168,165,167,221,30,195,245,61.950000,treatment_group
2,39,2025-03-18,17,control,363,486,452,45,4,74,68,5882.259388,265,181,18,5,150,150,115,77,49,1,1,6,125,94,125,94,0,0,0,330,324,4809.30,2149,505,665,2097,526,2097,957,499,492,498,619,34,459,819,244.416667,control_group
3,39,2025-03-19,3,treatment,42,71,62,12,3,17,15,932.700000,26,13,1,0,13,34,26,15,11,0,0,0,12,10,12,10,0,0,0,34,34,625.08,339,76,122,336,82,336,175,76,76,76,119,6,99,104,28.000000,treatment_group
4,39,2025-03-19,17,treatment,359,493,448,60,7,74,69,6108.709630,309,198,12,6,123,158,107,93,53,1,0,3,143,97,142,96,0,0,1,353,347,5849.66,2225,524,669,2175,555,2178,929,521,502,502,625,52,424,859,259.000000,treatment_group


In [5]:
##评估结果将自动输出到该目录下的Result文件夹中
import warnings
warnings.filterwarnings("ignore")
vcm = vcm_result(data=df,evaluate_dict=eval_dict)
vcm.run()

  0%|          | 0/27 [00:00<?, ?it/s]

39



  0%|          | 0/11 [00:00<?, ?it/s]

冒泡数



  9%|▉         | 1/11 [00:00<00:06,  1.46it/s]

发单数


... (640 outputs left out)


 91%|█████████ | 10/11 [00:14<00:01,  1.37s/it]

API完单数



100%|██████████| 27/27 [06:52<00:00, 15.29s/it]


,指标,城市,对照组,实验组,P值,绝对提升,相对提升,绝对提升置信区间下界,绝对提升置信区间上界
0,冒泡数,整体,362493.000000,363882.472181,4.700736e-02,1389.472181,0.003833,1.835193e+01,2.760592e+03
1,发单数,整体,252816.857143,253687.223545,1.275463e-01,870.366402,0.003443,-2.491129e+02,1.989846e+03
2,完单数,整体,170453.857143,170530.456082,6.868090e-01,76.598939,0.000449,-2.957669e+02,4.489648e+02
3,应答率,整体,0.901731,0.897670,1.010187e-05,-0.004060,-0.004503,NaN,NaN
4,成交率,整体,0.674219,0.674522,7.057282e-01,0.000303,0.000449,2.350936e-04,3.708646e-04
5,应答后司机取消率(含改派),整体,0.089672,0.088948,6.945631e-02,-0.000724,-0.008072,-3.781208e-02,3.636442e-02
6,应答后司机取消率(不含改派),整体,0.010791,0.011025,3.783149e-02,0.000233,0.021617,-4.364253e-03,4.830801e-03
7,应答后乘客取消率,整体,0.158577,0.156567,2.509965e-05,-0.002010,-0.012672,-6.729438e-02,6.327536e-02
8,自营完单数,整体,61057.285714,61474.124434,1.520105e-03,416.838720,0.006827,1.591753e+02,6.745022e+02
9,宏鹄完单数,整体,53585.000000,52487.735874,2.220446e-16,-1097.264126,-0.020477,-1.358638e+03,-8.358898e+02
